# State

In Haskell, **`State` is a newtype wrapper for a function** that represents a stateful computation. It is defined as:

```haskell
newtype State s a = State { runState :: s -> (a, s) }
````

This means a `State` computation is essentially a function that takes an **initial state (`s`)** and returns a tuple containing a **result value (`a`)** and a **new state value (`s`)**. 

### Why We Need State
In a pure functional language like Haskell, there is no global mutable state. To track changing information across several steps, you normally have to **manually pass the state value** from one function to the next and return the updated version each time. This process is known as **chaining state**, and it becomes **tedious and error-prone** because you must manually ensure that the "new" state from one step is the "input" state for the next.

The `State` monad is used to **abstract away this manual plumbing**. It allows you to write code that looks like it is mutating a variable, while under the hood, Haskell is still purely passing the state through each function for you. It is commonly used for:
*   **Random number generators**.
*   **Games** and **solvers**.
*   **Carrying working memory** while traversing complex data structures.

---

### Simple Example: Rolling Dice
Generating random numbers is a classic example of why `State` is useful. To get a "random" number, you need a generator (the state). Every time you get a number, the generator changes.

#### The Tedious Way (Manual Chaining)
Without the `State` monad, you have to manually track the generator (`s`) at every step:
```haskell
rollDieThreeTimes :: (Die, Die, Die)
rollDieThreeTimes = do
  let s = mkStdGen 0
  (d1, s1) = randomR (1, 6) s   -- Use s, get new state s1
  (d2, s2) = randomR (1, 6) s1  -- Use s1, get new state s2
  (d3, _)  = randomR (1, 6) s2  -- Use s2
  (intToDie d1, intToDie d2, intToDie d3)
```
In this manual approach, if you accidentally used `s` instead of `s1` for the second roll, you would get the same number twice.

#### The `State` Way (Automatic Chaining)
Using the `State` monad, you define an action that handles the state implicitly:
```haskell
rollDie :: State StdGen Die
rollDie = state $ \s -> 
  let (n, nextS) = randomR (1, 6) s 
  in (intToDie n, nextS)
```
Now, you can roll three dice without ever mentioning the generator variables again. Tools like `replicateM` or `liftA3` can **automatically thread the state** through the actions for you:
```haskell
-- Rolls 3 dice and handles all state chaining behind the scenes
threeRolls :: State StdGen [Die]
threeRolls = replicateM 3 rollDie
```
You can then run this computation using **`evalState`** (to get just the result) or **`runState`** (to get the result and the final generator).

## Exercises: Roll Your Own
1. Refactor `rollsToGetTwenty` into having the limit be a function argument.

```haskell
rollsToGetN :: Int -> StdGen -> Int
rollsToGetN = undefined
```

**Answer:**

```haskell
rollsToGetN :: Int -> StdGen -> Int
rollsToGetN n g = go 0 0 g
    where
        go :: Int -> Int -> StdGen -> Int
        go sum count gen
            | sum >= n = count
            | otherwise =
                let (die, nextGen) =
                    randomR (1, 6) gen
                in  go (sum + die) (count + 1) nextGen
```

2. Change `rollsToGetN` to recording the series of die that occurred in addition to the count.

```haskell
rollsCountLogged :: Int -> StdGen -> (Int, [Die])
rollsCountLogged n g = undefined
```

**Answer:**

```haskell
rollsCountLogged :: Int -> StdGen -> (Int, [Die])
rollsCountLogged n g = go 0 0 g
    where
        go :: Int -> Int -> [Die] -> StdGen -> Int
        go sum count dies gen
            | sum >= n = (count, dies)
            | otherwise =
                let 
                    (die, nextGen) = randomR (1, 6) gen
                in
                    go (sum + die) (count + 1) (dies ++ [intToDie die]) nextGen
```

## Write State for yourself

In [4]:
newtype MyState s a = MyState { runMyState :: s -> (a, s) }

### State Functor

Implement the Functor instance for State.

In [5]:
instance Functor (MyState s) where
    fmap :: (a -> b) -> MyState s a -> MyState s b
    fmap f myState = MyState g
        where
            g s = (f a, newS)
                where
                    (a, newS) = runMyState myState s

In [6]:
runMyState ((+1) <$> (MyState $ \s -> (0, s))) 0

Line 1: Move brackets to avoid $
Found:
(+ 1) <$> (MyState $ \ s -> (0, s))
Why not:
(+ 1) <$> MyState (\ s -> (0, s))Line 1: Use tuple-section
Found:
\ s -> (0, s)
Why not:
(0,)

(1,0)

### State Applicative
Write the Applicative instance for State.

In [15]:
instance Applicative (MyState s) where
    pure :: a -> MyState s a
    pure x = MyState (x,)

    (<*>) :: MyState s (a -> b) -> MyState s a -> MyState s b
    (<*>)  (MyState f) (MyState g) = MyState h
        where
            h s = (b, s_2)
                where
                    (f_ab, s_1) = f s
                    (a, s_2)    = g s_1
                    b           = f_ab a

### State Monad
Write the Monad instance for State.

In [18]:
instance Monad (MyState s) where
    return = pure

    (>>=) :: MyState s a -> (a -> MyState s b) -> MyState s b
    (>>=) (MyState f) f_aSb = MyState g
        where
            g s = (b, s_2)
                where
                    (a, s_1) = f s
                    stateSB  = f_aSb a
                    (b, s_2) = runMyState stateSB s_1